# 02 — Expressions & cast: column lineage с выражениями

Выходная колонка строится из выражения над одной или несколькими **входными колонками одного датасета** (без join'ов — это будет в 04). OpenLineage пометит зависимости как `DIRECT` type c subtype `TRANSFORMATION`.

> 📖 Per [OpenLineage spec](https://openlineage.io/docs/spec/facets/dataset-facets/column_lineage_facet) валидные subtypes:
> - `DIRECT`: `IDENTITY`, `TRANSFORMATION`, `AGGREGATION`
> - `INDIRECT`: `JOIN`, `GROUP_BY`, `FILTER`, `SORT`, `WINDOW`, `CONDITIONAL`
>
> Subtype `EXPRESSION` не существует — это собирательный термин в разговоре, а в JSON-фасете будет `TRANSFORMATION`.

**Что построим:** `default.stg_customers_enriched` из `raw_customers`:
- `id` — identity (subtype `IDENTITY`)
- `contact` = `concat(name, ' <', email, '>')` — две входные колонки → одна выходная (subtype `TRANSFORMATION`)
- `name_upper` = `upper(name)` — одна вход → одна выход, subtype `TRANSFORMATION`
- `email_domain` = `split(email, '@')[1]` — string func, subtype `TRANSFORMATION`
- `country_lower` = `lower(country)` — subtype `TRANSFORMATION`
- `id_str` = `cast(id AS STRING)` — type cast, subtype `TRANSFORMATION`

Все expressions из **одного** input dataset — column lineage будет чистым, без шума от join'ов.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_02_expressions')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_02_expressions', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:08:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:08:33 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:08:42 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:08:42 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:08:42 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_02_expressions


In [2]:
df = spark.sql('''
    SELECT
        id,
        concat(name, ' <', email, '>')  AS contact,
        upper(name)                     AS name_upper,
        split(email, '@')[1]            AS email_domain,
        lower(country)                  AS country_lower,
        cast(id AS STRING)              AS id_str
    FROM default.raw_customers
''')
df.write.mode('overwrite').saveAsTable('default.stg_customers_enriched')
spark.table('default.stg_customers_enriched').show(5, truncate=False)

26/05/27 13:08:45 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:08:45 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:08:45 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:08:45 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
26/05/27 13:08:48 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+---+-----------------------------+----------+------------+-------------+------+
|id |contact                      |name_upper|email_domain|country_lower|id_str|
+---+-----------------------------+----------+------------+-------------+------+
|50 |user_50 <user_50@example.com>|USER_50   |example.com |de           |50    |
|51 |user_51 <user_51@example.com>|USER_51   |example.com |fr           |51    |
|52 |user_52 <user_52@example.com>|USER_52   |example.com |ru           |52    |
|53 |user_53 <user_53@example.com>|USER_53   |example.com |us           |53    |
|54 |user_54 <user_54@example.com>|USER_54   |example.com |de           |54    |
+---+-----------------------------+----------+------------+-------------+------+
only showing top 5 rows



In [3]:
spark.stop()

## Что смотреть в Marquez

- У `contact` — **две** входящие стрелки (из `raw_customers.name` и `raw_customers.email`), subtype `TRANSFORMATION`.
- У `name_upper`, `email_domain`, `country_lower`, `id_str` — по одной стрелке, subtype `TRANSFORMATION` (а не `IDENTITY`, потому что прошла функция/cast).
- `id` без выражения → subtype `IDENTITY`.